# 2026-09-01_2 Refined subclass AUCell 打分与绘图
signature = refined 后 DM/DAL/AAM（celltype × direction，全方向；中间类不建 signature）
打分：胚胎/成年对象各自独立 ranking（不跨期比较），aucMaxRank = 5% 基因数
绘图：fovea+macula 合并为 central；boxplot + violin；x 轴 age-tissue 与 stage-tissue 两版；
      胚胎/成年分图，同一 celltype×subclass×direction panel 跨期锁定相同 y 轴范围


In [ ]:
suppressPackageStartupMessages({
  library(Seurat)
  library(AUCell)
  library(DelayedArray)
  library(matrixStats)
  library(SummarizedExperiment)
  library(dplyr)
  library(tidyr)
  library(readr)
  library(ggplot2)
})
set.seed(1)

date_str <- "2026-09-01_2"
data_dir <- "/project/work/2026-09-01_DEG_summary/Data/"
plot_dir <- "/project/work/2026-09-01_DEG_summary/Plot/"

refined_file <- file.path(data_dir, "2026-09-01_1_five_stage_trajectory_refined.csv")
seurat_files <- c(
  Embryonic = "/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds",
  Adult = "/project/work/2026-05-25_create_summary/Data/2026-06-05_new_adult_human_harmony_anno.rds"
)

celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
subclass_order <- c(
  "Developmental maintained",
  "Developmental aging-lost",
  "Adult acquired and maintained"
)
direction_order <- c("Macular direction", "Peripheral direction")
age_order <- c(
  "70days", "79days", "87days", "91days", "100days", "103days",
  "116days", "137days", "141days", "142days", "162days", "165days",
  "21years", "25years", "30years", "32years",
  "67years", "70years", "75years", "86years"
)
stage_order <- c("Early", "Mid", "Late", "Young", "Old")
tissue_order <- c("central", "peripheral")
cols.tissue.merged <- c(central = "#6A3D9A", peripheral = "#FFFF99")

auc_max_rank_pct <- 0.05
n_cores <- 4L

# 服务器兼容版 AUCell ranking（安装的 DelayedArray 与 AUCell_buildRankings 不兼容，必须保留）
.AUCell_buildRankings_new <- function(expr_mat) {
  expr_delayed <- DelayedArray::DelayedArray(-expr_mat)
  rankings <- do.call(cbind, DelayedArray::blockApply(
    expr_delayed,
    FUN = function(x, ...) matrixStats::colRanks(x, ..., useNames = TRUE),
    ties.method = "random", preserveShape = TRUE,
    grid = DelayedArray::colAutoGrid(expr_delayed)
  ))
  rownames(rankings) <- rownames(expr_mat)
  colnames(rankings) <- colnames(expr_mat)
  names(dimnames(rankings)) <- c("genes", "cells")
  new("aucellResults",
      SummarizedExperiment::SummarizedExperiment(assays = list(ranking = rankings)),
      nGenesDetected = numeric(0))
}


In [ ]:
# ---- 1. 用 refined 分类表构建 signature -------------------------------------
signature_key <- read_csv(refined_file, show_col_types = FALSE) %>%
  filter(
    celltype %in% celltype_order,
    subclass_refined %in% subclass_order,
    direction_class %in% direction_order
  ) %>%
  mutate(
    subclass_short = recode(subclass_refined,
      "Developmental maintained" = "DM",
      "Developmental aging-lost" = "DAL",
      "Adult acquired and maintained" = "AAM"
    ),
    direction_short = recode(direction_class,
      "Macular direction" = "Macular",
      "Peripheral direction" = "Peripheral"
    ),
    signature_id = paste(celltype, subclass_short, direction_short, sep = "__")
  ) %>%
  distinct(celltype, subclass = subclass_refined, direction_class, signature_id, gene) %>%
  group_by(celltype, subclass, direction_class, signature_id) %>%
  summarise(
    signature_genes = list(sort(unique(gene))),
    n_signature_genes = n(),
    signature_gene_names = paste(sort(unique(gene)), collapse = ";"),
    .groups = "drop"
  )
stopifnot(anyDuplicated(signature_key$signature_id) == 0, nrow(signature_key) > 0)

write_csv(
  signature_key %>% select(-signature_genes),
  file.path(data_dir, paste0(date_str, "_AUCell_signature_definition.csv"))
)
cat("signature 数：", nrow(signature_key), "\n")


In [ ]:
# ---- 2. 两期各自独立 AUCell 打分 --------------------------------------------
meta_list <- list()
qc_list <- list()
for (period in names(seurat_files)) {
  cat("[", format(Sys.time(), "%H:%M:%S"), "] 读取", period, "对象...\n")
  seurat_obj <- readRDS(seurat_files[[period]])
  expr_mat <- tryCatch(
    SeuratObject::LayerData(seurat_obj, assay = "RNA", layer = "counts"),
    error = function(e) SeuratObject::GetAssayData(seurat_obj, assay = "RNA", slot = "counts")
  )
  meta_df <- seurat_obj[[]]
  rm(seurat_obj); gc()
  stopifnot(identical(colnames(expr_mat), rownames(meta_df)))

  gene_sets <- lapply(signature_key$signature_genes, intersect, x = rownames(expr_mat))
  names(gene_sets) <- signature_key$signature_id
  qc_list[[period]] <- signature_key %>%
    select(-signature_genes) %>%
    mutate(
      period = period,
      n_detected_signature_genes = lengths(gene_sets),
      detected_gene_fraction = n_detected_signature_genes / n_signature_genes
    )

  cat("[", format(Sys.time(), "%H:%M:%S"), "] 构建", period, "ranking（独立，不跨期）...\n")
  rankings <- .AUCell_buildRankings_new(expr_mat)
  auc <- AUCell::AUCell_calcAUC(
    gene_sets[lengths(gene_sets) > 0], rankings,
    aucMaxRank = ceiling(auc_max_rank_pct * nrow(expr_mat)), nCores = n_cores
  )
  auc_mat <- t(AUCell::getAUC(auc))
  colnames(auc_mat) <- paste0("AUCell_", colnames(auc_mat))
  meta_list[[period]] <- cbind(meta_df, auc_mat[rownames(meta_df), , drop = FALSE])
  saveRDS(meta_list[[period]],
          file.path(data_dir, paste0(date_str, "_AUCell_", period, "_meta_data.rds")))
  rm(expr_mat, rankings, auc, auc_mat); gc()
  cat("[", format(Sys.time(), "%H:%M:%S"), "]", period, "打分完成。\n")
}
write_csv(bind_rows(qc_list), file.path(data_dir, paste0(date_str, "_AUCell_signature_QC.csv")))


In [ ]:
# ---- 3. 整理长表：合并 central、标注 stage ----------------------------------
plot_long <- bind_rows(lapply(names(meta_list), function(period) {
  meta_df <- meta_list[[period]]
  tissue_raw <- if ("tissue" %in% colnames(meta_df)) meta_df$tissue else meta_df$tissue_group
  meta_df %>%
    mutate(
      period = period,
      donor_age = as.character(donor_age),
      celltype = as.character(celltype),
      tissue = case_when(
        tolower(tissue_raw) %in% c("fovea", "macula", "central") ~ "central",
        tolower(tissue_raw) == "peripheral" ~ "peripheral",
        TRUE ~ NA_character_
      ),
      stage = case_when(
        donor_age %in% c("70days", "79days", "87days", "91days") ~ "Early",
        donor_age %in% c("100days", "103days", "116days") ~ "Mid",
        donor_age %in% c("137days", "141days", "142days", "162days", "165days") ~ "Late",
        donor_age %in% c("21years", "25years", "30years", "32years") ~ "Young",
        donor_age %in% c("67years", "70years", "75years", "86years") ~ "Old"
      )
    ) %>%
    filter(celltype %in% celltype_order, !is.na(tissue), !is.na(stage)) %>%
    select(period, celltype, donor_age, stage, tissue, starts_with("AUCell_")) %>%
    pivot_longer(starts_with("AUCell_"), names_to = "signature_id", values_to = "AUCell_score") %>%
    mutate(signature_id = sub("^AUCell_", "", signature_id)) %>%
    filter(!is.na(AUCell_score))
})) %>%
  left_join(
    signature_key %>% select(signature_id, signature_celltype = celltype, subclass, direction_class),
    by = "signature_id"
  ) %>%
  filter(celltype == signature_celltype) %>%   # 只保留 signature 对应 celltype 自身的细胞
  mutate(
    celltype = factor(celltype, levels = celltype_order),
    subclass = factor(subclass, levels = subclass_order),
    direction_class = factor(direction_class, levels = direction_order),
    tissue = factor(tissue, levels = tissue_order),
    stage = factor(stage, levels = stage_order),
    age_tissue = factor(paste(donor_age, tissue, sep = "\n"),
                        levels = paste(rep(age_order, each = 2), tissue_order, sep = "\n")),
    stage_tissue = factor(paste(stage, tissue, sep = "\n"),
                          levels = paste(rep(stage_order, each = 2), tissue_order, sep = "\n"))
  )

group_summary <- plot_long %>%
  group_by(period, stage, donor_age, tissue, celltype, signature_id, subclass, direction_class) %>%
  summarise(
    n_cells = n(),
    mean_AUCell_score = mean(AUCell_score),
    median_AUCell_score = median(AUCell_score),
    .groups = "drop"
  )
write_csv(group_summary, file.path(data_dir, paste0(date_str, "_AUCell_group_summary.csv")))

# 跨期共享 y 轴范围：每个 signature（= celltype × subclass × direction）取两期合并的 min/max
ylim_df <- plot_long %>%
  group_by(signature_id, celltype, subclass, direction_class) %>%
  summarise(ymin = min(AUCell_score), ymax = max(AUCell_score), .groups = "drop") %>%
  pivot_longer(c(ymin, ymax), values_to = "AUCell_score") %>%
  select(-name)


In [ ]:
# ---- 4. 绘图：2 期 × 2 x 轴版本 × 2 图型 = 8 个 PDF -------------------------
for (period in names(seurat_files)) {
  for (x_var in c("age_tissue", "stage_tissue")) {
    for (plot_type in c("boxplot", "violin")) {
      pdf_file <- file.path(plot_dir, paste0(
        date_str, "_AUCell_", period, "_", x_var, "_", plot_type, ".pdf"
      ))
      pdf(pdf_file, width = 16, height = 10, useDingbats = FALSE)
      for (one_subclass in subclass_order) {
        for (one_direction in direction_order) {
          one_df <- plot_long %>%
            filter(period == !!period, subclass == one_subclass, direction_class == one_direction)
          if (nrow(one_df) == 0) next
          one_ylim <- ylim_df %>%
            filter(subclass == one_subclass, direction_class == one_direction,
                   signature_id %in% one_df$signature_id)
          p <- ggplot(one_df, aes(x = .data[[x_var]], y = AUCell_score, fill = tissue)) +
            geom_blank(data = one_ylim, aes(x = NULL, y = AUCell_score, fill = NULL))
          p <- if (plot_type == "boxplot") {
            p + geom_boxplot(width = 0.68, outlier.shape = NA, linewidth = 0.25)
          } else {
            p + geom_violin(scale = "width", linewidth = 0.25, trim = TRUE)
          }
          p <- p +
            facet_wrap(~celltype, ncol = 3, scales = "free_y", drop = FALSE) +
            scale_fill_manual(values = cols.tissue.merged, drop = FALSE) +
            labs(
              title = paste(period, "AUCell |", one_subclass, "|", one_direction),
              subtitle = "Refined signatures; independent per-period ranking (scores not comparable across periods); y-range locked across periods per panel.",
              x = gsub("_", " + ", x_var), y = "AUCell score", fill = "Tissue"
            ) +
            theme_classic(base_size = 10) +
            theme(
              axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 7),
              strip.text = element_text(face = "bold"),
              plot.title = element_text(face = "bold", hjust = 0.5),
              plot.subtitle = element_text(hjust = 0.5, size = 8),
              panel.grid = element_blank()
            )
          print(p)
        }
      }
      dev.off()
      cat("已输出：", pdf_file, "\n")
    }
  }
}


In [ ]:
# ---- 5. 参数记录 ------------------------------------------------------------
writeLines(
  c(
    paste0("date_str: ", date_str),
    paste0("refined_file: ", refined_file),
    paste0("seurat_files: ", paste(names(seurat_files), seurat_files, sep = "=", collapse = " | ")),
    paste0("auc_max_rank_pct: ", auc_max_rank_pct),
    "tissue merge: fovea+macula -> central",
    "ranking: independent per period; scores not comparable across periods",
    "y-axis: locked per celltype x subclass x direction panel across the two period figures",
    "",
    capture.output(sessionInfo())
  ),
  file.path(data_dir, paste0(date_str, "_AUCell_parameters_sessionInfo.txt"))
)
cat("全部完成。\n")


## 批量运行日志（Rscript 后台执行记录）
```
signature 数： 31 
[ 18:28:48 ] 读取 Embryonic 对象...
[ 18:30:40 ] 构建 Embryonic ranking（独立，不跨期）...
Using 4 cores with doMC.
[ 18:44:57 ] Embryonic 打分完成。
[ 18:44:57 ] 读取 Adult 对象...
[ 18:48:32 ] 构建 Adult ranking（独立，不跨期）...
Using 4 cores with doMC.
[ 19:03:30 ] Adult 打分完成。
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Embryonic_age_tissue_boxplot.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Embryonic_age_tissue_violin.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Embryonic_stage_tissue_boxplot.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Embryonic_stage_tissue_violin.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Adult_age_tissue_boxplot.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Adult_age_tissue_violin.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Adult_stage_tissue_boxplot.pdf 
已输出： /project/work/2026-09-01_DEG_summary/Plot//2026-09-01_2_AUCell_Adult_stage_tissue_violin.pdf 
Warning messages:
1: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 
2: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 
3: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 
4: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 
5: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 
全部完成。
```
